In [42]:
import os, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
matplotlib.rcParams["font.family"] = "sans-serif"
matplotlib.rcParams["svg.fonttype"] = "none"
import matplotlib.pyplot as plt
from adjustText import adjust_text

In [43]:
INPUT_CSV = "./outputs/protein_difference_summary.csv"
OUT_DIR   = "./outputs"
FIG_DPI   = 300
FONT_SIZE = 8

In [44]:
COL_NS  = "#CCCCCC"
COL_NEG = "#2166AC"
COL_POS = "#D6604D"
COL_REF = "orange"   # dark red

In [45]:
SIG_P   = 0.01
Y_CAP   = 15
REF_KEY = "EGFP-NLS_vs_ATXN1"

In [46]:
LABEL_LIST   = ["MEF2A","MEF2C","TPD52","TPD52L2","RRAGB","MEAF6","FBXO25",
                "CTNND1","KCNT2","VAV2","ATG13","ASAP1","PPP3CA","PPFIA3"]
ALWAYS_LABEL = ["ERC1", "AP1S2"]

df = pd.read_csv(INPUT_CSV, index_col=0)

PANELS = [
    ("granularity_auc_cohens_d", "granularity_auc_pvalue", "Granularity"),
    ("morans_i_auc_cohens_d",    "morans_i_auc_pvalue",    "Moran\u2019s I"),
]

fig, axes = plt.subplots(1, 2, figsize=(9, 4.5))
fig.subplots_adjust(wspace=0.45)

for ax, (d_col, p_col, title) in zip(axes, PANELS):

    sub = df[[d_col, p_col]].dropna().copy()
    sub["neglogp"] = -np.log10(sub[p_col].clip(lower=1e-300))
    is_ref = sub.index == REF_KEY

    sig     = (~is_ref) & (sub[p_col] < SIG_P)
    sig_neg = sig & (sub[d_col] < 0)
    sig_pos = sig & (sub[d_col] > 0)
    ns      = ~sig & ~is_ref

    above    = sub["neglogp"] > Y_CAP
    sub_plot = sub.copy()
    sub_plot.loc[above, "neglogp"] = Y_CAP

    labeled_names = set(p for p in LABEL_LIST if p in sub_plot.index and sig.loc[p])
    labeled_names |= set(p for p in ALWAYS_LABEL if p in sub_plot.index)
    if REF_KEY in sub_plot.index:
        labeled_names.add(REF_KEY)

    # ── Scatter: unlabeled first, labeled on top with black outline ────────────
    for labeled in [False, True]:
        lw = 0.4 if labeled else 0
        ec = "black" if labeled else "none"
        zo_off = 1 if labeled else 0

        def _sc(mask, color, size, alpha, zorder):
            m_sel = mask & (sub_plot.index.isin(labeled_names) if labeled
                            else ~sub_plot.index.isin(labeled_names))
            for cap, marker in [(False, "o"), (True, "^")]:
                m = m_sel & (above if cap else ~above)
                if m.any():
                    ax.scatter(sub_plot.loc[m, d_col], sub_plot.loc[m, "neglogp"],
                               c=color, s=size, marker=marker, alpha=alpha,
                               edgecolors=ec, linewidths=lw,
                               zorder=zorder + zo_off)

        _sc(ns,      COL_NS,  12, 0.7, 2)
        _sc(sig_neg, COL_NEG, 18, 0.9, 3)
        _sc(sig_pos, COL_POS, 18, 0.9, 3)

    if REF_KEY in sub_plot.index:
        ref_marker = "^" if above.loc[REF_KEY] else "o"
        ax.scatter(sub_plot.loc[REF_KEY, d_col],
                   sub_plot.loc[REF_KEY, "neglogp"],
                   c=COL_REF, s=18, marker=ref_marker,
                   edgecolors="black", linewidths=0.4, zorder=6)

    # ── Threshold lines ────────────────────────────────────────────────────────
    ax.axhline(-np.log10(SIG_P), color="#888888", lw=0.6, linestyle="--", zorder=1)
    ax.axvline(0,                color="#888888", lw=0.6, linestyle=":",  zorder=1)

    # ── Symmetric x-axis ──────────────────────────────────────────────────────
    xabs = max(abs(sub_plot[d_col].max()), abs(sub_plot[d_col].min())) * 1.08
    ax.set_xlim(-xabs, xabs)
    ax.set_ylim(bottom=-0.5, top=Y_CAP * 1.3)

    # ── Labels: place all, then iteratively push overlapping ones up ───────────
    to_label = [p for p in LABEL_LIST if p in sub_plot.index and sig.loc[p]]
    to_label += [p for p in ALWAYS_LABEL if p in sub_plot.index]
    if REF_KEY in sub_plot.index:
        to_label.append(REF_KEY)

    # Initial text positions: slightly above each dot
    label_info = []
    for name in to_label:
        px = sub_plot.loc[name, d_col]
        py = sub_plot.loc[name, "neglogp"]
        if name == REF_KEY:
            lcolor = COL_REF
            display = "EGFP/ATXN1"
        elif (name in ALWAYS_LABEL) and (not sig.get(name, False)):
            lcolor = "#888888"
            display = name
        else:
            lcolor = "black"
            display = name
        label_info.append({"name": name, "display": display, "px": px, "py": py,
                            "tx": px, "ty": py + 0.4, "color": lcolor})

    # Draw all texts first (needed for bbox)
    texts = []
    for info in label_info:
        t = ax.text(info["tx"], info["ty"], info["display"],
                    fontsize=FONT_SIZE, color=info["color"],
                    ha="center", va="bottom", zorder=7)
        texts.append(t)

    # Iterative push: sort by ty, push overlapping labels upward
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    STEP = 0.7
    for _ in range(30):
        moved = False
        bbs = [t.get_window_extent(renderer) for t in texts]
        for i in range(len(texts)):
            for j in range(i + 1, len(texts)):
                if bbs[i].overlaps(bbs[j]):
                    # Push the higher one up
                    hi = i if texts[i].get_position()[1] >= texts[j].get_position()[1] else j
                    x, y = texts[hi].get_position()
                    texts[hi].set_position((x, y + STEP))
                    bbs[hi] = texts[hi].get_window_extent(renderer)
                    moved = True
        if not moved:
            break

    # Draw connectors from final text position to dot
    for t, info in zip(texts, label_info):
        tx, ty = t.get_position()
        ax.annotate("", xy=(info["px"], info["py"]),
                    xytext=(tx, ty),
                    arrowprops=dict(arrowstyle="-", color="#AAAAAA", lw=0.4),
                    zorder=6)

    # ── Axis style ─────────────────────────────────────────────────────────────
    ax.set_xlabel("Cohen\u2019s d  (S \u2212 L)", fontsize=FONT_SIZE)
    ax.set_ylabel("\u2212log\u2081\u2080(p)",      fontsize=FONT_SIZE)
    ax.set_title(title, fontsize=FONT_SIZE + 1, fontweight="bold")

    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.8)
    ax.spines["bottom"].set_linewidth(0.8)
    ax.tick_params(axis="both", direction="out", length=3, width=0.8, labelsize=FONT_SIZE)

    n_sig = sig.sum()
    n_exp = (~is_ref).sum()
    ax.text(0.97, 0.03, f"{n_sig}/{n_exp} sig",
            transform=ax.transAxes, fontsize=FONT_SIZE, va="bottom", ha="right",
            color="#555555")

for ext, dpi in [("png", FIG_DPI), ("svg", FIG_DPI)]:
    out = os.path.join(OUT_DIR, f"fig_volcano.{ext}")
    fig.savefig(out, dpi=dpi, bbox_inches="tight", facecolor="white")
    print(f"Saved {out}")
plt.close(fig)
print("Done.")

Saved ./outputs/fig_volcano.png
Saved ./outputs/fig_volcano.svg
Done.
